# **Guía Práctica de Machine Learning: Preprocesamiento y Optimización**
##### *Tutoría - Marta Harana*

Dudas a resolver:
1. Cuándo y cómo separar los datos en Train/Test y variables (X, y).
2. Diferencias prácticas entre `pd.get_dummies` y `OneHotEncoder`.
3. Aplicación de `GridSearchCV` vs `RandomizedSearchCV`.


## Importación de librerías y creación de datos sintéticos

In [12]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    train_test_split,
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

import warnings
warnings.filterwarnings('ignore')
pd.options.mode.copy_on_write = True

X_raw, y_raw = make_classification(
    n_samples=1000, 
    n_features=4, 
    n_informative=3, 
    n_redundant=0, 
    random_state=42
)
df = pd.DataFrame(X_raw, columns=["Num1", "Num2", "Num3", "Num4"])

# Añadimos una columna categórica para simular datos reales
np.random.seed(42) 
categorias = ["Bajo", "Medio", "Alto"]
df["Categoria"] = np.random.choice(categorias, size=1000)
df["Target"] = y_raw

print("Primeras filas del dataset original:")
df.head()


Primeras filas del dataset original:


,Num1,Num2,Num3,Num4,Categoria,Target
0,1.170199,-1.110199,0.790988,-0.910081,Alto,0
1,0.198680,1.499221,0.004205,0.988123,Bajo,1
2,1.695723,0.126734,0.069711,0.222349,Alto,1
3,-0.531455,-0.169270,-1.100362,-2.224787,Alto,0
4,0.194052,-2.479343,0.297014,1.518522,Bajo,0


## Split de Datos (X, y y Train/Test)

### 1. ¿Cuándo separar en X e y, y cuándo hacer Train/Test Split?

**Regla de oro:** El split de entrenamiento y prueba (Train/Test) se debe hacer **antes** de cualquier exploración y transformación de datos (como escalado u One-Hot Encoding). Si transformas todo el dataset junto, información del conjunto de test se "filtrará" en el de entrenamiento (*data leakage*), falseando las métricas de evaluación.

**Mi recomendación**:
- Separar, en primer lugar, en `train_set` y `test_set`
- EDA en `train_set` -> Selección de features más importantes (en el caso de un problema supervisado) comparándolo contra el TARGET
- División en `X_train`, `X_test`, `y_train` e `y_test`
- Aplicación de transformación, únicamente en el conjunto de entrenamiento.

#### ¿Se puede dividir desde un principio en `X_train`, `X_test`, `y_train` e `y_test`?

Sí, pero se tiene que tener en cuenta de que las features y el target están separados para poder hacer cualquier tipo de combinación entre ambos.

In [ ]:
train_set, test_set = train_test_split(df, test_size=0.2, stratify=df['Target'], random_state=42)
print(f"Dimensiones de train_set: {train_set.shape}") # Matriz --> DataFrame (2 dimesiones)
print(f"Dimensiones de test_set: {test_set.shape}") # Matriz --> DataFrame (2 dimesiones)

Dimensiones de train_set: (800, 6)
Dimensiones de test_set: (200, 6)


In [ ]:
# Paso 1: Separar en X (características) e y (variable objetivo)
X = df.drop(columns=["Target"])
y = df["Target"]

# Paso 2: Dividir en Train y Test (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dimensiones de X_train: {X_train.shape} | X_test: {X_test.shape}") # Matrices --> DataFrame (2 dimesiones)
print(f"Dimensiones de y_train: {y_train.shape} | y_test: {y_test.shape}")# Arrays --> DataSerie (1 dimensión)

Dimensiones de X_train: (800, 5) | X_test: (200, 5)
Dimensiones de y_train: (800,) | y_test: (200,)


## Codificación Categórica (`pd.get_dummies` vs `OneHotEncoder`)

-  `pd.get_dummies`: Modifica el DataFrame directamente. Es ideal para Análisis Exploratorio de Datos (EDA), pero peligroso en producción porque no "recuerda" las categorías aprendidas.
-  `OneHotEncoder`: Guarda el estado. Aprende las categorías del conjunto de entrenamiento (`.fit`) y las aplica de forma idéntica al conjunto de prueba o producción (`.transform`), gestionando categorías nuevas o ausentes de forma segura.


In [7]:
# ENFOQUE 1: pd.get_dummies (Solo para exploración rápida)
# Si lo aplicamos por separado, corremos el riesgo de tener columnas diferentes en train y test si falta alguna categoría.
X_train_dummies = pd.get_dummies(X_train, columns=["Categoria"])
print(
    f"Columnas con pd.get_dummies en Train: {X_train_dummies.shape[1]} columnas"
)


# ENFOQUE 2: OneHotEncoder (El estándar correcto para Machine Learning)
# 1. Instanciamos el encoder configurando cómo manejar categorías desconocidas en producción
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# 2. Ajustamos Y transformamos el conjunto de ENTRENAMIENTO (aprende las categorías de Train)
cat_train_encoded = encoder.fit_transform(X_train[["Categoria"]])

# 3. SOLO transformamos el conjunto de TEST (usa las categorías aprendidas de Train)
cat_test_encoded = encoder.transform(X_test[["Categoria"]])

# Creamos DataFrames limpios para continuar con el modelo
column_names = encoder.get_feature_names_out(["Categoria"])

X_train_final = pd.concat(
    [
        X_train.drop(columns=["Categoria"]).reset_index(drop=True),
        pd.DataFrame(cat_train_encoded, columns=column_names),
    ],
    axis=1,
)

X_test_final = pd.concat(
    [
        X_test.drop(columns=["Categoria"]).reset_index(drop=True),
        pd.DataFrame(cat_test_encoded, columns=column_names),
    ],
    axis=1,
)

print("Estructura final lista para el modelo:")
X_train_final.head(2)

Columnas con pd.get_dummies en Train: 7 columnas
Estructura final lista para el modelo:


,Num1,Num2,Num3,Num4,Categoria_Alto,Categoria_Bajo,Categoria_Medio
0,-0.945746,-0.361007,1.399681,-1.208712,1.0,0.0,0.0
1,-0.215610,1.320788,0.453216,1.082914,0.0,0.0,1.0


In [15]:
column_names

array(['Categoria_Alto', 'Categoria_Bajo', 'Categoria_Medio'],
      dtype=object)

In [14]:
cat_train_encoded

array([[1., 0., 0.],
       [0., 0., 1.],
       [0., 0., 1.],
       ...,
       [0., 0., 1.],
       [0., 0., 1.],
       [1., 0., 0.]])

## Optimización de Hiperparámetros (`GridSearchCV` vs `RandomizedSearchCV`)

Usaremos un modelo `RandomForestClassifier` para comparar ambos métodos de optimización:
-   `GridSearchCV`: Evaluará exhaustivamente la multiplicación de todas las opciones.
-   `RandomizedSearchCV`: Evaluará únicamente el número de iteraciones fijadas aleatoriamente (`n_iter`), ahorrando tiempo de cómputo en espacios grandes.

In [13]:
# Definimos el modelo base
rf = RandomForestClassifier(random_state=42, verbose=1)

# CONFIGURACIÓN 1: GridSearchCV (Búsqueda Exhaustiva)
# Espacio de búsqueda pequeño y definido
grid_param = {
    "n_estimators": [10, 50],
    "max_depth": [None, 5, 10],
    "min_samples_split": [1, 10],
}

grid_search = GridSearchCV(
    estimator=rf, param_grid=grid_param, cv=3, scoring="accuracy", n_jobs=-1
)
grid_search.fit(X_train_final, y_train)

print(
    f"GridSearchCV - Mejor puntuación: {grid_search.best_score_:.4f}"
)
print(f"GridSearchCV - Mejores parámetros: {grid_search.best_params_}\n")


# CONFIGURACIÓN 2: RandomizedSearchCV (Búsqueda Aleatoria)
# Espacio de búsqueda continuo o muy amplio
random_param = {
    "n_estimators": randint(10, 200),
    "max_depth": [None, 3, 5, 10, 20],
    "min_samples_split": randint(2, 11),
}

# Controlamos el tiempo limitando el número de intentos con `n_iter`
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=random_param,
    n_iter=10,
    cv=3,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
)
random_search.fit(X_train_final, y_train)

print(
    f"RandomizedSearchCV - Mejor puntuación: {random_search.best_score_:.4f}"
)
print(f"RandomizedSearchCV - Mejores parámetros: {random_search.best_params_}")

GridSearchCV - Mejor puntuación: 0.9337
GridSearchCV - Mejores parámetros: {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 10}

RandomizedSearchCV - Mejor puntuación: 0.9363
RandomizedSearchCV - Mejores parámetros: {'max_depth': 10, 'min_samples_split': 7, 'n_estimators': 98}


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
